# Pipeline Pan-Genome Bacterien — Correction

**Auteur : Marwa Zidi** — Universite Paris Cite

---

## Organisation des donnees

| Dossier | Contenu |
|---------|--------|
| `/root/data/for-Pangenome /Raw_Files/` | Reads FASTQ bruts (R1/R2) |
| `/root/data/for-Pangenome /Inputs/` | Assemblages pre-calcules |
| `/root/results/` | Resultats |

---

# PHASE 1 — Controle qualite et Assemblage

---

## Etape 1.1 — Explorer les donnees

In [ ]:
ls -lh /root/data/for-Pangenome\ /Raw_Files/

In [ ]:
ls -lh /root/data/for-Pangenome\ /Inputs/

In [ ]:
mkdir -p /root/results/01_quality_control
mkdir -p /root/results/02_trimmed
mkdir -p /root/results/03_assembly
mkdir -p /root/results/04_annotation
mkdir -p /root/results/05_pangenome
echo "Repertoires crees"

## Etape 1.2 — Controle qualite avec FastQC

**Variable SAMPLE** : a adapter selon le nom reel de vos fichiers FASTQ.

In [ ]:
# Variables — adaptez selon vos noms de fichiers reels
SAMPLE="strain1"

R1="/root/data/for-Pangenome /Raw_Files/${SAMPLE}_R1.fastq.gz"
R2="/root/data/for-Pangenome /Raw_Files/${SAMPLE}_R2.fastq.gz"

OUTPUT_DIR="/root/results/01_quality_control"

echo "Echantillon : $SAMPLE"
echo "R1 : $R1"
echo "R2 : $R2"

In [ ]:
fastqc -o $OUTPUT_DIR $R1 $R2

echo "Rapports FastQC generes dans $OUTPUT_DIR"

## Etape 1.3 — Nettoyage avec Trimmomatic

**Parametres** :
- `ILLUMINACLIP` : supprime les adaptateurs Nextera
- `LEADING:3 TRAILING:3` : supprime bases de debut/fin si qualite < 3
- `SLIDINGWINDOW:4:15` : fenetre de 4 bases, qualite moyenne >= 15
- `MINLEN:36` : supprime reads < 36 pb

In [ ]:
R1_OUT="/root/results/02_trimmed/${SAMPLE}_R1_paired.fastq.gz"
R1_UNPAIRED="/root/results/02_trimmed/${SAMPLE}_R1_unpaired.fastq.gz"
R2_OUT="/root/results/02_trimmed/${SAMPLE}_R2_paired.fastq.gz"
R2_UNPAIRED="/root/results/02_trimmed/${SAMPLE}_R2_unpaired.fastq.gz"

echo "R1 paired   : $R1_OUT"
echo "R2 paired   : $R2_OUT"

In [ ]:
trimmomatic PE -phred33 \
    $R1 $R2 \
    $R1_OUT $R1_UNPAIRED \
    $R2_OUT $R2_UNPAIRED \
    ILLUMINACLIP:/usr/share/trimmomatic/NexteraPE-PE.fa:2:30:10 \
    LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

echo "Reads nettoyes"
ls -lh /root/results/02_trimmed/

## Etape 1.4 — Assemblage avec SPAdes

> Un assemblage pre-calcule est disponible dans `/root/data/for-Pangenome /Inputs/`

In [ ]:
R1_CLEAN="/root/results/02_trimmed/${SAMPLE}_R1_paired.fastq.gz"
R2_CLEAN="/root/results/02_trimmed/${SAMPLE}_R2_paired.fastq.gz"
OUTPUT_DIR="/root/results/03_assembly/$SAMPLE"

echo "Assemblage de : $SAMPLE"

In [ ]:
spades.py -1 $R1_CLEAN -2 $R2_CLEAN \
    -o $OUTPUT_DIR \
    --careful \
    -t 4

echo "Assemblage termine : ${OUTPUT_DIR}/contigs.fasta"
ls -lh ${OUTPUT_DIR}/contigs.fasta

## Etape 1.5 — Validation avec QUAST

In [ ]:
CONTIGS="/root/results/03_assembly/$SAMPLE/contigs.fasta"
OUTPUT_DIR="/root/results/03_assembly/${SAMPLE}_quast"

echo "Validation de : $CONTIGS"

In [ ]:
quast.py $CONTIGS -o $OUTPUT_DIR

echo "Rapport QUAST dans ${OUTPUT_DIR}/report.html"

In [ ]:
cat ${OUTPUT_DIR}/report.txt

---

# PHASE 2 — Identification et Annotation

---

## Etape 2.1 — Typage MLST avec pubMLST

1. https://pubmlst.org/ → selectionner l'espece
2. Uploader `contigs.fasta` : `/root/results/03_assembly/${SAMPLE}/contigs.fasta`
3. Lancer l'analyse → noter le Sequence Type (ST)
4. Sauvegarder dans `/root/results/04_annotation/`

## Etape 2.2 — Annotation avec Prokka

Adaptez `GENUS` et `SPECIES` selon votre bacterie.

In [ ]:
CONTIGS="/root/results/03_assembly/$SAMPLE/contigs.fasta"
OUTPUT_DIR="/root/results/04_annotation/${SAMPLE}_prokka"

# Adaptez selon votre bacterie
GENUS="Escherichia"
SPECIES="coli"

echo "Annotation de : $SAMPLE ($GENUS $SPECIES)"

In [ ]:
prokka --outdir $OUTPUT_DIR \
    --prefix $SAMPLE \
    --kingdom Bacteria \
    --genus $GENUS \
    --species $SPECIES \
    --strain $SAMPLE \
    --cpus 4 \
    --force \
    $CONTIGS

echo "Fichier GFF genere : ${OUTPUT_DIR}/${SAMPLE}.gff"

In [ ]:
cat ${OUTPUT_DIR}/${SAMPLE}.txt

## Etape 2.3 — antiSMASH (outil web)

1. https://antismash.secondarymetabolites.org/
2. Uploader : `/root/results/04_annotation/${SAMPLE}_prokka/${SAMPLE}.gbk`
3. Cocher KnownClusterBlast, ClusterBlast, SubClusterBlast
4. Lancer (10-30 min) → telecharger dans `/root/results/04_annotation/`

## Etape 2.4 — enrichR (optionnel)

In [ ]:
GFF_FILE="/root/results/04_annotation/${SAMPLE}_prokka/${SAMPLE}.gff"
GENES_FILE="/root/results/04_annotation/genes_resistance.txt"

grep -i "resistance" $GFF_FILE | cut -f9 | cut -d';' -f1 | sed 's/ID=//g' > $GENES_FILE

echo "Genes de resistance extraits :"
cat $GENES_FILE

In [ ]:
bash /root/enrichr_shell.sh -f $GENES_FILE -o /root/results/04_annotation/enrichr_results

echo "Resultats dans enrichr_results/enrichr_results.csv"

---

# PHASE 3 — Pan-Genome

---

## Etape 3.1 — Genomes de reference NCBI

1. https://www.ncbi.nlm.nih.gov/genome/ → rechercher l'espece
2. Selectionner 5-10 genomes representatifs → telecharger FASTA ou GFF
3. Sauvegarder dans `/root/results/05_pangenome/reference_genomes/`

In [ ]:
mkdir -p /root/results/05_pangenome/reference_genomes

In [ ]:
# Annoter un genome de reference si vous avez un FASTA
REF_GENOME="/root/results/05_pangenome/reference_genomes/GCF_000005845.fasta"
REF_NAME="E_coli_K12"
OUTPUT_DIR="/root/results/05_pangenome/reference_genomes/${REF_NAME}_prokka"

prokka --outdir $OUTPUT_DIR \
    --prefix $REF_NAME \
    --kingdom Bacteria \
    --genus Escherichia \
    --species coli \
    --cpus 4 \
    --force \
    $REF_GENOME

echo "GFF de reference : ${OUTPUT_DIR}/${REF_NAME}.gff"

## Etape 3.2 — Preparer les GFF pour Roary

In [ ]:
mkdir -p /root/results/05_pangenome/all_gffs

In [ ]:
# Copier les GFF de vos souches assemblees
cp /root/results/04_annotation/strain1_prokka/strain1.gff /root/results/05_pangenome/all_gffs/
cp /root/results/04_annotation/strain2_prokka/strain2.gff /root/results/05_pangenome/all_gffs/
cp /root/results/04_annotation/strain3_prokka/strain3.gff /root/results/05_pangenome/all_gffs/

echo "GFF des souches copies"

In [ ]:
# Copier les GFF de reference
cp /root/results/05_pangenome/reference_genomes/*_prokka/*.gff /root/results/05_pangenome/all_gffs/

echo "GFF de reference copies"

In [ ]:
ls -lh /root/results/05_pangenome/all_gffs/

NB_GFF=$(ls /root/results/05_pangenome/all_gffs/*.gff 2>/dev/null | wc -l)
echo "Nombre de fichiers GFF : $NB_GFF"

if [ $NB_GFF -lt 3 ]; then
    echo "ATTENTION : Roary necessite au moins 3 genomes !"
else
    echo "Nombre de GFF suffisant pour Roary"
fi

## Etape 3.3 — Pan-genome avec Roary

**Options** :
- `-e -n` : alignement MAFFT des genes core
- `-p 4` : 4 threads
- `-v` : mode verbeux

**Fichiers de sortie cles** :
- `gene_presence_absence.csv` : matrice core/accessoire
- `summary_statistics.txt` : statistiques du pan-genome
- `accessory_binary_genes.fa.newick` : arbre phylogenetique
- `core_gene_alignment.aln` : alignement des genes core

In [ ]:
GFF_DIR="/root/results/05_pangenome/all_gffs"
OUTPUT_DIR="/root/results/05_pangenome/roary_output"

echo "GFF disponibles : $(ls $GFF_DIR/*.gff | wc -l)"
echo "Sortie : $OUTPUT_DIR"

In [ ]:
roary -f $OUTPUT_DIR \
    -e -n \
    -p 4 \
    -v \
    $GFF_DIR/*.gff

echo "Analyse pan-genome terminee"

In [ ]:
cat ${OUTPUT_DIR}/summary_statistics.txt

In [ ]:
head -n 5 ${OUTPUT_DIR}/gene_presence_absence.csv | cut -d',' -f1-5

## Etape 3.4 — Visualisation avec Phandango

1. https://jameshadfield.github.io/phandango/
2. Uploader :
   - **Arbre** : `results/05_pangenome/roary_output/accessory_binary_genes.fa.newick`
   - **Matrice** : `results/05_pangenome/roary_output/gene_presence_absence.csv`
3. Explorer : zoom, filtres, cliquer sur les genes
4. Sauvegarder en SVG/PNG

---

## Checklist finale — Correction

**Phase 1** : FastQC + Trimmomatic + SPAdes + QUAST

**Phase 2** : pubMLST + Prokka (GFF) + antiSMASH

**Phase 3** : >= 3 GFF + Roary + Phandango

---

**Pipeline pan-genome complet.**